# Train GPT-BERT on native (non-translated) Hindi/Telugu data

Trains monolingual GPT-BERT models from scratch on the CC-100-derived native data (`pulipakav-1/hi-te`), using this project's existing `babybabellm-gptbert` training code (`Babylm2026/gpt-bert/{hindi,telugu}`), scaled down to fit a single Colab GPU in a few hours instead of the original multi-GPU cluster-scale run (`max_steps=15625`, `global_batch_size=32768` across 3-4 GPUs).

**Runs both Hindi and Telugu sequentially, one after the other, in a single pass** -- just run all cells top to bottom.

Steps per language: download native data -> train a fresh tokenizer on it -> shard/tokenize the data -> train the model -> (optional) push the checkpoint to Hugging Face.

In [ ]:
# Cell 1: clone the repo once and install dependencies for both languages
!git clone https://github.com/vishnup22/BabyLM.git
%cd BabyLM
!git checkout evaluation
!git pull
!pip install -q -r Babylm2026/gpt-bert/hindi/requirements.txt
!pip install -q -r Babylm2026/gpt-bert/telugu/requirements.txt

In [ ]:
# Cell 2: log in to Hugging Face (only needed if you want the optional push-to-HF step at
# the end -- pulipakav-1/hi-te itself is a public dataset, no token needed just to read it)
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # use whatever secret name you saved your token under
!hf auth whoami

In [ ]:
# Cell 3: the full per-language pipeline, as a function -- called once per language below.
# Uses subprocess with an explicit cwd per step instead of %cd, so the two languages'
# runs don't interfere with each other's working directory state.
import subprocess
from pathlib import Path
from huggingface_hub import hf_hub_download, HfApi

REPO_ROOT = Path.cwd()

# Hyperparameters -- SCALED DOWN from the original cluster-scale run (max_steps=15625,
# global_batch_size=32768 across 3-4 GPUs) to fit a single Colab GPU in a few hours.
# Watch the first ~50 steps' pace (printed by the training loop) and adjust MAX_STEPS if
# you want a longer or shorter run. Model architecture is unchanged (configs/base.json).
MAX_STEPS = 2000
GLOBAL_BATCH_SIZE = 256
LOCAL_BATCH_SIZE = 64  # lower this if you hit an out-of-memory error on a smaller GPU
SEQ_LENGTH = 128


def run(cmd, cwd):
    print(f"\n$ (cwd={cwd}) {' '.join(cmd)}\n")
    subprocess.run(cmd, cwd=cwd, check=True)


def train_language(lang):
    lang_code = {"hindi": "hi", "telugu": "te"}[lang]
    dataset_name = f"native-{lang}"
    lang_dir = REPO_ROOT / "Babylm2026" / "gpt-bert" / lang

    print(f"\n{'=' * 70}\nTraining {lang}\n{'=' * 70}")

    # 1. Download native data and lay it out where the tooling expects it
    src_path = hf_hub_download(repo_id="pulipakav-1/hi-te", filename=f"{lang}.txt", repo_type="dataset")
    raw_dir = lang_dir / "data" / "raw" / dataset_name
    raw_dir.mkdir(parents=True, exist_ok=True)
    dest_path = raw_dir / f"{dataset_name}.train.{lang_code}.txt"
    dest_path.write_bytes(Path(src_path).read_bytes())
    print(f"Placed {dest_path} ({dest_path.stat().st_size:,} bytes)")

    # 2. Train a fresh tokenizer on the native data (not reusing the old translated-data
    #    tokenizer, which would reintroduce the tokenizer-granularity issue found earlier)
    run(["python", "tools/train_tokenizer_local.py",
         "--dataset", dataset_name,
         "--data_root", "data/raw",
         "--output", "tokenizers/tokenizer_native_16384.json",
         "--vocab_size", "16384"], cwd=lang_dir)

    # 3. Tokenize and shard (2% held out for validation)
    run(["python", "tools/prepare_local_shards.py",
         "--dataset", dataset_name,
         "--data_root", "data/raw",
         "--tokenizer", "tokenizers/tokenizer_native_16384.json",
         "--output_base", "data/processed",
         "--valid_fraction", "0.02"], cwd=lang_dir)

    # 4. Train, single GPU
    os.environ["WANDB_MODE"] = "disabled"  # no W&B account needed
    run(["python", "train_single_gpu.py",
         "--train_path", "../data/processed/train",
         "--valid_path", "../data/processed/valid",
         "--config_file", "../configs/base.json",
         "--tokenizer_path", "../tokenizers/tokenizer_native_16384.json",
         "--name", f"native-{lang}-gptbert",
         "--output_dir", "../model_checkpoints",
         "--hybrid_numerator", "2",
         "--hybrid_denominator", "3",
         "--global_batch_size", str(GLOBAL_BATCH_SIZE),
         "--local_batch_size", str(LOCAL_BATCH_SIZE),
         "--seq_length", str(SEQ_LENGTH),
         "--max_steps", str(MAX_STEPS),
         "--save_every", "200",
         "--validate_every", "0",
         "--seed", "42"], cwd=lang_dir / "pretraining")

    print(f"\nFinished training {lang}. Checkpoints in {lang_dir / 'model_checkpoints'}")


def push_language(lang):
    lang_dir = REPO_ROOT / "Babylm2026" / "gpt-bert" / lang
    repo_id = f"pulipakav-1/native-{lang}-gptbert"
    api = HfApi()
    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    for local_path, repo_path in [
        (lang_dir / "model_checkpoints" / f"native-{lang}-gptbert_2_3_ema.bin", "model_ema.bin"),
        (lang_dir / "tokenizers" / "tokenizer_native_16384.json", "tokenizer.json"),
        (lang_dir / "configs" / "base.json", "config_base.json"),
    ]:
        api.upload_file(path_or_fileobj=str(local_path), path_in_repo=repo_path, repo_id=repo_id, repo_type="model")
    print(f"Pushed to https://huggingface.co/{repo_id}")

In [ ]:
# Cell 4: run both languages sequentially, one after the other
for lang in ["hindi", "telugu"]:
    train_language(lang)

## Optional: push both checkpoints to Hugging Face

Requires the HF login cell above to have been run with a write-scoped token.

In [ ]:
# Cell 5 (optional): push both trained checkpoints
for lang in ["hindi", "telugu"]:
    push_language(lang)